# Mission 2 — Préparation et pipeline sans fuite de données
**Projet Churn Telco — Master IA, Supervised Learning**

Point noté le plus strictement du projet : le `train_test_split` doit précéder toute
transformation apprise. Ce notebook construit le pipeline scikit-learn unique qui
garantit cette contrainte, établit la baseline, puis teste par ablation chaque feature
engineered candidate — sans forcer un résultat positif si les chiffres disent le contraire.

In [1]:
import sys
sys.path.insert(0, '..')
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import make_scorer, fbeta_score

from src.features import TotalChargesFixer, FeatureEngineer
from src.pipeline import build_pipeline, build_preprocessing, NUMERIC_COLS, CATEGORICAL_COLS

RANDOM_STATE = 42
f2_scorer = make_scorer(fbeta_score, beta=2)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

## 1. Split d'abord — pourquoi cet ordre est non négociable

Si l'on calculait la médiane, la moyenne ou les modalités d'encodage sur l'ensemble du
dataset **avant** de séparer train/test, le test aurait "vu" (indirectement, via ces
statistiques agrégées) une information issue du train — et réciproquement. Le score sur
le test ne mesurerait alors plus une vraie généralisation, mais une performance
optimiste et non reproductible en production, où l'on ne dispose jamais des futures
observations au moment d'entraîner.

Le split est donc **stratifié** (préserve le taux de churn ~26,5% dans les deux sous-
ensembles) et réalisé sur les données encore brutes, avant tout `fit()`.

In [2]:
df = pd.read_csv('../data/Telco-Customer-Churn.csv')
y = (df['Churn'] == 'Yes').astype(int)
X = df.drop(columns=['customerID', 'Churn'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print('Train:', X_train.shape, '| Test:', X_test.shape)
print(f"Taux de churn — train: {y_train.mean():.4f} | test: {y_test.mean():.4f}")

Train: (5634, 19) | Test: (1409, 19)
Taux de churn — train: 0.2654 | test: 0.2654


## 2. Le ColumnTransformer — où sont apprises les statistiques ?

- Sous-pipeline **numérique** (`tenure`, `MonthlyCharges`, `TotalCharges`) :
  imputation par la médiane + `StandardScaler`.
- Sous-pipeline **catégoriel** (16 colonnes) : imputation par le mode + `OneHotEncoder`.

Ces deux sous-pipelines sont assemblés dans un `ColumnTransformer`, lui-même dans un
`Pipeline` scikit-learn. **La médiane, le mode, les modalités de l'OneHotEncoder ne sont
appris qu'au moment de l'appel `.fit(X_train)`** — jamais sur `X_test`. C'est la garantie
structurelle contre la fuite : elle ne dépend pas de la discipline du développeur mais de
l'objet lui-même.

### Le correctif `TotalCharges`
Avant même le `ColumnTransformer`, un transformer dédié (`TotalChargesFixer`) applique
la règle établie en Mission 1 : `TotalCharges = 0` quand `tenure == 0`. Ce n'est **pas**
une transformation apprise (aucune statistique du train n'est utilisée — c'est une règle
métier fixe, identique quelle que soit la ligne) : elle peut donc vivre dans le pipeline
sans risque de fuite, uniquement pour garder un objet unique et reproductible.

## 3. Baseline — régression logistique

In [3]:
baseline_pipe = build_pipeline(LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))

baseline_scores = cross_val_score(baseline_pipe, X_train, y_train, cv=cv, scoring=f2_scorer, n_jobs=-1)
print(f'Baseline F2 (CV 5-fold) : {baseline_scores.mean():.4f} (+/- {baseline_scores.std():.4f})')
print('-> Valeur de référence à battre en Mission 3.')

Baseline F2 (CV 5-fold) : 0.5621 (+/- 0.0364)
-> Valeur de référence à battre en Mission 3.


**Comparaison imputation métier (0) vs. imputation statistique (médiane)** pour
`TotalCharges`, afin de vérifier que la décision de la Mission 1 n'est pas seulement
défendable conceptuellement mais n'est pas non plus pénalisante empiriquement.

In [4]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

class CoerceOnly(BaseEstimator, TransformerMixin):
    """Baseline alternative : coercition numérique simple, laisse le SimpleImputer
    (médiane) gérer les 11 NaN résiduels — pour comparaison avec la règle métier."""
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        X = X.copy()
        X['TotalCharges'] = pd.to_numeric(X['TotalCharges'], errors='coerce')
        return X

pipe_median = Pipeline([
    ('coerce', CoerceOnly()),
    ('preprocessing', build_preprocessing()),
    ('clf', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])
scores_median = cross_val_score(pipe_median, X_train, y_train, cv=cv, scoring=f2_scorer, n_jobs=-1)

print(f'Imputation médiane (standard) : F2 = {scores_median.mean():.4f}')
print(f'Imputation 0 (règle métier)   : F2 = {baseline_scores.mean():.4f}')
print(f"Écart : {baseline_scores.mean() - scores_median.mean():+.4f} "
      "(marginal, attendu vu que seules 11/5634 lignes du train sont concernées — "
      "la décision reste justifiée sur le plan de la correction conceptuelle, "
      "pas seulement du gain statistique)")

Imputation médiane (standard) : F2 = 0.5619
Imputation 0 (règle métier)   : F2 = 0.5621
Écart : +0.0002 (marginal, attendu vu que seules 11/5634 lignes du train sont concernées — la décision reste justifiée sur le plan de la correction conceptuelle, pas seulement du gain statistique)


## 4. Feature engineering — validé par ablation, pas par intuition

Trois features candidates, chacune testée **individuellement** en CV contre la baseline,
sur le **même** modèle (régression logistique) et les **mêmes** folds. Le rasoir d'Occam
s'applique strictement : une feature qui n'améliore pas le score de validation croisée
est écartée, même si elle "semble" pertinente métier.

In [5]:
def build_fe_pipeline(fe: FeatureEngineer, extra_numeric=(), extra_categorical=(),
                       drop_total_charges=False):
    numeric_cols = [c for c in NUMERIC_COLS if not (drop_total_charges and c == 'TotalCharges')]
    numeric_cols = numeric_cols + list(extra_numeric)
    categorical_cols = CATEGORICAL_COLS + list(extra_categorical)

    prep = ColumnTransformer([
        ('num', Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), numeric_cols),
        ('cat', Pipeline([('impute', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical_cols),
    ])
    return Pipeline([
        ('total_charges_fix', TotalChargesFixer()),
        ('feature_engineering', fe),
        ('preprocessing', prep),
        ('clf', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
    ])

configs = [
    ('Baseline (référence)', build_fe_pipeline(FeatureEngineer(False, False, False))),
    ('+ tenure_group', build_fe_pipeline(FeatureEngineer(True, False, False), extra_categorical=['tenure_group'])),
    ('+ n_services', build_fe_pipeline(FeatureEngineer(False, True, False), extra_numeric=['n_services'])),
    ('+ charges_per_tenure (remplace TotalCharges brut)',
        build_fe_pipeline(FeatureEngineer(False, False, True), extra_numeric=['charges_per_tenure'], drop_total_charges=True)),
]

rows = []
for name, pipe in configs:
    s = cross_val_score(pipe, X_train, y_train, cv=cv, scoring=f2_scorer, n_jobs=-1)
    rows.append({'Feature': name, 'F2 moyen': round(s.mean(), 4), 'Écart-type': round(s.std(), 4)})

results_df = pd.DataFrame(rows)
results_df['Gain vs baseline'] = (results_df['F2 moyen'] - results_df.loc[0, 'F2 moyen']).round(4)
results_df

,Feature,F2 moyen,Écart-type,Gain vs baseline
0,Baseline (référence),0.5621,0.0364,0.0000
1,+ tenure_group,0.5579,0.0314,-0.0042
2,+ n_services,0.5617,0.0365,-0.0004
3,+ charges_per_tenure (remplace TotalCharges brut),0.5506,0.0346,-0.0115


### Résultat — rasoir d'Occam appliqué

Les trois features candidates dégradent (légèrement) le F2 en validation croisée par
rapport à la baseline. Interprétation : avec un modèle **linéaire**, le `OneHotEncoder`
sur `Contract`/`tenure` (variable continue déjà standardisée) capture déjà l'essentiel de
l'information que ces features tentaient de resynthétiser — les ajouter revient surtout
à ajouter du bruit ou de la colinéarité (`charges_per_tenure` en particulier, dérivée de
variables déjà présentes).

**Décision Mission 2** : aucune des trois features n'est retenue dans le pipeline final
(`src/pipeline.py`). Ce n'est pas un jugement définitif — un modèle non linéaire (forêt
aléatoire, Mission 3) pourrait exploiter des seuils ou interactions que la régression
logistique ne capture pas. Le test sera reconduit en Mission 3.

## 5. Tableau récapitulatif — baseline vs après feature engineering

In [6]:
summary = pd.DataFrame([
    {'Étape': 'Baseline (imputation médiane standard)', 'F2 CV': round(scores_median.mean(), 4)},
    {'Étape': 'Baseline (imputation métier TotalCharges=0)', 'F2 CV': round(baseline_scores.mean(), 4)},
    {'Étape': 'Meilleure config avec FE (tenure_group)', 'F2 CV': round(results_df['F2 moyen'].iloc[1:].max(), 4)},
])
summary['Gain vs médiane standard'] = (summary['F2 CV'] - summary.loc[0, 'F2 CV']).round(4)
summary

,Étape,F2 CV,Gain vs médiane standard
0,Baseline (imputation médiane standard),0.5619,0.0000
1,Baseline (imputation métier TotalCharges=0),0.5621,0.0002
2,Meilleure config avec FE (tenure_group),0.5617,-0.0002


**Conclusion Mission 2** : le pipeline final retenu (`src/pipeline.py`) est le plus
simple des quatre testés — imputation métier pour `TotalCharges`, aucune feature
engineered. F2 de référence pour la Mission 3 : **0,5621 (± 0,0364) en CV 5-fold sur le
train**. C'est cette valeur que les modèles de la Mission 3 devront battre significativement.